# Wikimedia Dump Converter

This Colab notebook installs the converter as a package and exposes a small widget UI for converting single Wikimedia dumps or dump catalog URLs.

In [ ]:
import re
import subprocess
import sys

GITHUB_REPO = "https://github.com/0xEodum/WikiParser.git"
INSTALL_S3_EXTRA = False

def clean_github_repo_url(value):
    text = str(value).strip()
    match = re.search(r"https://github\.com/[^\]\)\s]+", text)
    if match:
        text = match.group(0)
    text = text.rstrip("/.)]")
    if not text.startswith("https://github.com/"):
        raise ValueError(f"Expected a GitHub HTTPS URL, got: {value!r}")
    if not text.endswith(".git"):
        text = f"{text}.git"
    return text

repo_url = clean_github_repo_url(GITHUB_REPO)
spec = f"wiki-dump-converter[s3] @ git+{repo_url}" if INSTALL_S3_EXTRA else f"git+{repo_url}"
print(f"Installing from: {spec}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec, "ipywidgets"])

In [ ]:
import ipywidgets as widgets
from IPython.display import display

source = widgets.Text(
    value="https://dumps.wikimedia.org/other/mediawiki_content_current/enwiki/2026-05-01/xml/bzip2/",
    description="Source",
    layout=widgets.Layout(width="95%"),
)
output_dir = widgets.Text(value="/content/wiki_output", description="Output", layout=widgets.Layout(width="95%"))
pages_per_file = widgets.IntText(value=1000, description="Pages/file")
limit = widgets.IntText(value=10, description="Page limit")
archive_limit = widgets.IntText(value=1, description="Archive limit")
namespaces = widgets.Text(value="0", description="Namespaces")
lead_title = widgets.Text(value="Introduction", description="Lead title")
force_catalog = widgets.Checkbox(value=True, description="Treat as catalog")
include_redirects = widgets.Checkbox(value=False, description="Include redirects")
output_format = widgets.Dropdown(options=["json", "ontology"], value="json", description="Format")
storage = widgets.Dropdown(options=["local", "s3", "both"], value="local", description="Storage")
s3_bucket = widgets.Text(value="", description="S3 bucket", layout=widgets.Layout(width="70%"))
s3_prefix = widgets.Text(value="", description="S3 prefix", layout=widgets.Layout(width="70%"))
s3_endpoint = widgets.Text(value="", description="Endpoint", layout=widgets.Layout(width="70%"))
s3_region = widgets.Text(value="", description="Region", layout=widgets.Layout(width="70%"))
run_button = widgets.Button(description="Run conversion", button_style="primary")
out = widgets.Output()

display(widgets.VBox([
    source, output_dir,
    widgets.HBox([pages_per_file, limit, archive_limit]),
    widgets.HBox([namespaces, lead_title]),
    widgets.HBox([force_catalog, include_redirects]),
    widgets.HBox([output_format, storage]),
    widgets.Accordion(children=[widgets.VBox([s3_bucket, s3_prefix, s3_endpoint, s3_region])], titles=("S3 settings",)),
    run_button,
    out,
]))

In [ ]:
from pathlib import Path
import json

from wikiparser.dump_converter import convert_catalog, convert_dump, is_catalog_source, parse_namespaces
from wikiparser.s3_io import load_s3_config, upload_directory_to_s3

def optional_int(widget):
    return widget.value if widget.value and widget.value > 0 else None

def on_run_clicked(_button):
    out.clear_output()
    with out:
        kwargs = dict(
            output_dir=output_dir.value,
            pages_per_file=pages_per_file.value,
            lead_title=lead_title.value,
            namespaces=parse_namespaces(namespaces.value),
            skip_redirects=not include_redirects.value,
            limit=optional_int(limit),
            output_format=output_format.value,
        )
        if force_catalog.value or is_catalog_source(source.value):
            summary = convert_catalog(
                catalog_url=source.value,
                archive_limit=optional_int(archive_limit),
                **kwargs,
            )
            print(f"Wrote {summary.pages_written} pages from {summary.archives_processed}/{summary.archives_seen} archives")
        else:
            summary = convert_dump(source=source.value, **kwargs)
            print(f"Wrote {summary.pages_written} pages")

        if storage.value in {"s3", "both"}:
            config = load_s3_config(
                bucket=s3_bucket.value or None,
                prefix=s3_prefix.value,
                endpoint_url=s3_endpoint.value or None,
                region_name=s3_region.value or None,
            )
            s3_summary = upload_directory_to_s3(summary.output_dir, config)
            print(f"Uploaded {s3_summary.files_uploaded} files to s3://{s3_summary.bucket}/{s3_summary.prefix}")

        json_files = sorted(Path(summary.output_dir).rglob("*.json"))
        print(f"Output directory: {summary.output_dir}")
        print(f"JSON files: {len(json_files)}")
        if json_files:
            sample = json.loads(json_files[0].read_text(encoding="utf-8"))
            print(f"First JSON: {json_files[0]}")
            print(json.dumps(sample[0] if isinstance(sample, list) else sample, ensure_ascii=False, indent=2)[:1500])

run_button.on_click(on_run_clicked)

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/wiki_output", "zip", output_dir.value)
print(f"Created archive: {zip_path}")